In [0]:
# Configuration


CATALOG = "worldbank_ai"
BRONZE_SCHEMA = "bronze"

WORLD_BANK_API_BASE_URL = "https://api.worldbank.org/v2"

INDICATOR_METADATA_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.indicator_metadata_raw"
)

OBSERVATIONS_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.indicator_observations_raw"
)

START_YEAR = 2010

print(f"Start year: {START_YEAR}")
print(f"Metadata table: {INDICATOR_METADATA_TABLE}")
print(f"Target table: {OBSERVATIONS_TABLE}")

In [0]:
# Imports
import requests
import time

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Imports loaded.")

In [0]:
# Load only validated indicators


validated_indicator_df = (
    spark.table(INDICATOR_METADATA_TABLE)
    .filter(F.col("is_valid") == True)
    .select(
        "indicator_code",
        "indicator_name"
    )
)

display(validated_indicator_df)

validated_indicators = [
    row["indicator_code"]
    for row in validated_indicator_df.collect()
]

print(
    f"Validated indicators loaded: "
    f"{len(validated_indicators)}"
)

In [0]:
# Determine end year

CURRENT_YEAR = datetime.now(timezone.utc).year

END_YEAR = CURRENT_YEAR

print(f"Observation request range: {START_YEAR}:{END_YEAR}")

In [0]:
# API helper with retry


def get_json_with_retry(
    url,
    params=None,
    max_retries=5,
    timeout=60
):
    retryable_status_codes = {
        429, 500, 502, 503, 504
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in retryable_status_codes:

                retry_after = response.headers.get(
                    "Retry-After"
                )

                if retry_after:
                    try:
                        wait_seconds = int(retry_after)
                    except ValueError:
                        wait_seconds = min(
                            2 ** attempt,
                            60
                        )
                else:
                    wait_seconds = min(
                        2 ** attempt,
                        60
                    )

                print(
                    f"HTTP {response.status_code}. "
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

        except requests.RequestException as exc:

            if attempt == max_retries:
                raise RuntimeError(
                    f"Request failed after "
                    f"{max_retries} attempts."
                ) from exc

            wait_seconds = min(
                2 ** attempt,
                60
            )

            print(
                f"Request error: {exc}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        "World Bank API request failed unexpectedly."
    )

In [0]:
# Test one observation request

TEST_INDICATOR = "NY.GDP.MKTP.KD.ZG"

test_url = (
    f"{WORLD_BANK_API_BASE_URL}/country/all/"
    f"indicator/{TEST_INDICATOR}"
)

test_params = {
    "format": "json",
    "date": f"{START_YEAR}:{END_YEAR}",
    "per_page": 10,
    "page": 1
}

test_response = get_json_with_retry(
    test_url,
    params=test_params
)

print(f"Response type: {type(test_response)}")

if isinstance(test_response, list):
    print(f"Top-level elements: {len(test_response)}")

In [0]:
# Inspect response metadata


if (
    not isinstance(test_response, list)
    or len(test_response) < 2
):
    raise RuntimeError(
        "Unexpected World Bank observation response."
    )

test_metadata = test_response[0]
test_records = test_response[1] or []

print("Response metadata")
print("-" * 60)

for key, value in test_metadata.items():
    print(f"{key}: {value}")

print()
print(f"Records on test page: {len(test_records)}")

In [0]:
# Inspect one observation


if test_records:

    sample = test_records[0]

    print("Sample observation")
    print("-" * 60)

    for key, value in sample.items():
        print(f"{key}: {value}")

In [0]:
#Fetch all pages for one indicator


def fetch_indicator_observations(
    indicator_code,
    start_year,
    end_year,
    per_page=500
):

    url = (
        f"{WORLD_BANK_API_BASE_URL}/country/all/"
        f"indicator/{indicator_code}"
    )

    base_params = {
        "format": "json",
        "date": f"{start_year}:{end_year}",
        "per_page": per_page
    }

    first_response = get_json_with_retry(
        url,
        params={
            **base_params,
            "page": 1
        }
    )

    if (
        not isinstance(first_response, list)
        or len(first_response) < 2
    ):
        raise RuntimeError(
            f"Unexpected response structure for "
            f"{indicator_code}"
        )

    metadata = first_response[0]
    first_records = first_response[1] or []

    total_pages = int(
        metadata.get("pages", 1)
    )

    expected_total = int(
        metadata.get("total", len(first_records))
    )

    all_records = list(first_records)

    for page in range(2, total_pages + 1):

        response = get_json_with_retry(
            url,
            params={
                **base_params,
                "page": page
            }
        )

        if (
            not isinstance(response, list)
            or len(response) < 2
        ):
            raise RuntimeError(
                f"Unexpected response for "
                f"{indicator_code}, page {page}"
            )

        all_records.extend(
            response[1] or []
        )

    if len(all_records) != expected_total:
        raise RuntimeError(
            f"{indicator_code}: API expected "
            f"{expected_total} records but "
            f"{len(all_records)} were downloaded."
        )

    return all_records, metadata

In [0]:
#Test the function with GDP growth


test_all_records, test_all_metadata = (
    fetch_indicator_observations(
        TEST_INDICATOR,
        START_YEAR,
        END_YEAR
    )
)

print(
    f"{TEST_INDICATOR}: "
    f"{len(test_all_records)} records retrieved"
)

In [0]:
# Ingest all 15 indicators

all_observations = []
ingestion_summary = []

for index, indicator_code in enumerate(
    validated_indicators,
    start=1
):

    print(
        f"[{index}/{len(validated_indicators)}] "
        f"Fetching {indicator_code}..."
    )

    try:

        records, metadata = (
            fetch_indicator_observations(
                indicator_code,
                START_YEAR,
                END_YEAR
            )
        )

        all_observations.extend(records)

        non_null_count = sum(
            1
            for record in records
            if record.get("value") is not None
        )

        ingestion_summary.append({
            "indicator_code": indicator_code,
            "status": "SUCCESS",
            "records": len(records),
            "non_null_values": non_null_count,
            "source_last_updated": metadata.get("lastupdated"),
            "error": None
        })

        print(
            f"    {len(records):,} records | "
            f"{non_null_count:,} non-null values | "
            f"source updated: {metadata.get('lastupdated')}"
        )

    except Exception as exc:

        ingestion_summary.append({
            "indicator_code": indicator_code,
            "status": "FAILED",
            "records": 0,
            "non_null_values": 0,
            "source_last_updated": None,
            "error": str(exc)
        })

        print(
            f"    FAILED: {exc}"
        )

    # Small pause between indicators to avoid hammering the API
    time.sleep(0.25)

In [0]:
# Validation

from pyspark.sql import types as T

# Explicit schema prevents Spark inference errors
summary_schema = T.StructType([
    T.StructField("indicator_code", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("records", T.LongType(), False),
    T.StructField("non_null_values", T.LongType(), False),
    T.StructField("source_last_updated", T.StringType(), True),
    T.StructField("error", T.StringType(), True)
])

summary_df = spark.createDataFrame(
    ingestion_summary,
    schema=summary_schema
)

display(
    summary_df.orderBy("indicator_code")
)

failed_indicators = [
    item
    for item in ingestion_summary
    if item["status"] == "FAILED"
]

successful_count = (
    len(ingestion_summary)
    - len(failed_indicators)
)

print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)

print(
    f"Successful indicators: {successful_count}"
)

print(
    f"Failed indicators:     {len(failed_indicators)}"
)

print(
    f"Raw records:           {len(all_observations):,}"
)

if failed_indicators:

    print("\nFailed indicators:")

    for item in failed_indicators:
        print(
            f"{item['indicator_code']} -> "
            f"{item['error']}"
        )

    raise RuntimeError(
        "One or more indicators failed. "
        "Bronze write stopped to prevent "
        "an incomplete snapshot."
    )

print("\nAll indicators downloaded successfully.")
print("Safe to continue to Bronze transformation.")

In [0]:
# Check ingestion status

from pyspark.sql import types as T


# Explicit schema because the "error" column may contain
# only None values when every indicator succeeds.
summary_schema = T.StructType([
    T.StructField(
        "indicator_code",
        T.StringType(),
        False
    ),
    T.StructField(
        "status",
        T.StringType(),
        False
    ),
    T.StructField(
        "records",
        T.LongType(),
        False
    ),
    T.StructField(
        "non_null_values",
        T.LongType(),
        False
    ),
    T.StructField(
        "error",
        T.StringType(),
        True
    )
])


summary_df = spark.createDataFrame(
    ingestion_summary,
    schema=summary_schema
)


display(
    summary_df.orderBy(
        "indicator_code"
    )
)


failed_indicators = [
    item
    for item in ingestion_summary
    if item["status"] == "FAILED"
]


successful_indicators = (
    len(ingestion_summary)
    - len(failed_indicators)
)


print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)

print(
    f"Successful indicators: "
    f"{successful_indicators}"
)

print(
    f"Failed indicators: "
    f"{len(failed_indicators)}"
)

print(
    f"Raw observation records downloaded: "
    f"{len(all_observations):,}"
)


if failed_indicators:

    print("\nFailed indicators:")

    for item in failed_indicators:

        print(
            item["indicator_code"],
            "->",
            item["error"]
        )

    raise RuntimeError(
        "One or more indicators failed ingestion. "
        "Do not write an incomplete Bronze snapshot."
    )


print("\nValidation passed.")
print("All requested indicators were downloaded.")

In [0]:
# Define Bronze schema

from pyspark.sql import types as T

observation_schema = T.StructType([

    T.StructField(
        "entity_id",
        T.StringType(),
        True
    ),

    T.StructField(
        "entity_name",
        T.StringType(),
        True
    ),

    T.StructField(
        "entity_iso3_code",
        T.StringType(),
        True
    ),

    T.StructField(
        "indicator_id",
        T.StringType(),
        True
    ),

    T.StructField(
        "indicator_name",
        T.StringType(),
        True
    ),

    # Keep raw API representation in Bronze.
    # We will convert this to IntegerType in Silver.
    T.StructField(
        "year_raw",
        T.StringType(),
        True
    ),

    # Nullable because World Bank observations
    # can legitimately have missing values.
    T.StructField(
        "value",
        T.DoubleType(),
        True
    ),

    T.StructField(
        "unit",
        T.StringType(),
        True
    ),

    T.StructField(
        "observation_status",
        T.StringType(),
        True
    ),

    T.StructField(
        "decimal",
        T.LongType(),
        True
    ),

    # Pipeline lineage
    T.StructField(
        "source_system",
        T.StringType(),
        False
    ),

    T.StructField(
        "source_endpoint",
        T.StringType(),
        False
    ),

    T.StructField(
        "requested_start_year",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "requested_end_year",
        T.IntegerType(),
        False
    ),

    # World Bank API response metadata.
    # Nullable for this initial snapshot because
    # we did not retain metadata per indicator.
    T.StructField(
        "source_last_updated",
        T.StringType(),
        True
    ),

    T.StructField(
        "ingested_at",
        T.TimestampType(),
        False
    )
])

print("Bronze observation schema:")
print(observation_schema.simpleString())

In [0]:
# Flatten World Bank API observations into Bronze rows

from datetime import datetime, timezone


# Use one timestamp for the entire ingestion batch
ingestion_timestamp = datetime.now(timezone.utc)


bronze_rows = []


for record in all_observations:

    # Nested structures returned by World Bank API
    country = record.get("country") or {}
    indicator = record.get("indicator") or {}

    bronze_rows.append({

        # Entity information
        "entity_id": country.get("id"),

        "entity_name": country.get("value"),

        "entity_iso3_code": record.get(
            "countryiso3code"
        ),


        # Indicator information
        "indicator_id": indicator.get("id"),

        "indicator_name": indicator.get("value"),


        # Preserve year as raw string in Bronze
        "year_raw": (
            str(record.get("date"))
            if record.get("date") is not None
            else None
        ),


        # Preserve missing observations as NULL
        "value": (
            float(record.get("value"))
            if record.get("value") is not None
            else None
        ),


        # Source-provided fields
        "unit": record.get("unit"),

        "observation_status": record.get(
            "obs_status"
        ),

        "decimal": (
            int(record.get("decimal"))
            if record.get("decimal") is not None
            else None
        ),


        # Data lineage
        "source_system":
            "World Bank Indicators API",

        "source_endpoint":
            "https://api.worldbank.org/v2/"
            "country/all/indicator/{indicator}",

        "requested_start_year": START_YEAR,

        "requested_end_year": END_YEAR,

        # We did not retain this metadata for every
        # indicator during the current ingestion run.
        "source_last_updated": None,

        "ingested_at": ingestion_timestamp
    })


print(
    f"Raw API observations: "
    f"{len(all_observations):,}"
)

print(
    f"Flattened Bronze rows: "
    f"{len(bronze_rows):,}"
)


# The flattening operation must not lose records
if len(bronze_rows) != len(all_observations):

    raise RuntimeError(
        "Observation count changed during "
        "Bronze flattening."
    )


print("Bronze flattening validation passed.")

In [0]:
# Create Bronze Spark DataFrame using explicit schema

observations_df = spark.createDataFrame(
    bronze_rows,
    schema=observation_schema
)


print(
    f"Spark DataFrame rows: "
    f"{observations_df.count():,}"
)


# Validate row count
expected_count = len(bronze_rows)
actual_count = observations_df.count()

if actual_count != expected_count:

    raise RuntimeError(
        f"Row count mismatch. "
        f"Expected {expected_count:,}, "
        f"but Spark DataFrame contains "
        f"{actual_count:,}."
    )


print("Spark DataFrame row-count validation passed.")


# Inspect schema
observations_df.printSchema()


# Preview records
display(
    observations_df.limit(10)
)

In [0]:
# Validate year range

display(
    observations_df
    .groupBy("year_raw")
    .count()
    .orderBy("year_raw")
)

invalid_year_count = (
    observations_df
    .filter(
        F.col("year_raw").isNull()
        | ~F.col("year_raw").rlike(r"^\d{4}$")
    )
    .count()
)

out_of_range_year_count = (
    observations_df
    .filter(
        F.col("year_raw").cast("int").isNotNull()
        & (
            (F.col("year_raw").cast("int") < START_YEAR)
            | (F.col("year_raw").cast("int") > END_YEAR)
        )
    )
    .count()
)

print(
    f"Invalid year values: {invalid_year_count}"
)

print(
    f"Out-of-range year values: {out_of_range_year_count}"
)

if invalid_year_count > 0:
    raise RuntimeError(
        "Unexpected year values found in Bronze observations."
    )

if out_of_range_year_count > 0:
    raise RuntimeError(
        "Observation years fall outside the requested range."
    )

print("Year validation passed.")

In [0]:
# Measure data coverage by indicator

coverage_df = (
    observations_df
    .groupBy(
        "indicator_id",
        "indicator_name"
    )
    .agg(
        F.count("*").alias(
            "total_observations"
        ),

        F.count("value").alias(
            "non_null_observations"
        ),

        F.sum(
            F.when(
                F.col("value").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_observations"
        ),

        F.round(
            (
                F.count("value")
                / F.count("*")
            ) * 100,
            2
        ).alias(
            "coverage_pct"
        )
    )
    .orderBy(
        F.desc("coverage_pct")
    )
)

display(coverage_df)

In [0]:
# Check duplicate entity-indicator-year combinations

duplicate_keys_df = (
    observations_df
    .groupBy(
        "entity_id",
        "indicator_id",
        "year_raw"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_key_count = (
    duplicate_keys_df.count()
)

print(
    f"Duplicate entity-indicator-year keys: "
    f"{duplicate_key_count}"
)

if duplicate_key_count > 0:

    print(
        "Duplicate keys found. "
        "Inspect before continuing."
    )

    display(
        duplicate_keys_df.limit(20)
    )

    raise RuntimeError(
        "Duplicate observation keys detected. "
        "Bronze write stopped for investigation."
    )

print("Duplicate-key validation passed.")

In [0]:
# Check entity and indicator coverage

entity_count = (
    observations_df
    .select("entity_id")
    .where(
        F.col("entity_id").isNotNull()
    )
    .distinct()
    .count()
)

indicator_count = (
    observations_df
    .select("indicator_id")
    .where(
        F.col("indicator_id").isNotNull()
    )
    .distinct()
    .count()
)

print(
    f"Distinct entities: {entity_count}"
)

print(
    f"Distinct indicators: {indicator_count}"
)

if indicator_count != len(validated_indicators):

    raise RuntimeError(
        f"Expected {len(validated_indicators)} indicators, "
        f"but found {indicator_count}."
    )

print("Entity and indicator validation passed.")

In [0]:
# Final validation before Bronze write

expected_count = len(all_observations)

actual_count = observations_df.count()

null_indicator_ids = (
    observations_df
    .filter(
        F.col("indicator_id").isNull()
    )
    .count()
)

null_entity_ids = (
    observations_df
    .filter(
        F.col("entity_id").isNull()
    )
    .count()
)

print(f"Expected records:       {expected_count:,}")
print(f"DataFrame records:      {actual_count:,}")
print(f"Null indicator IDs:     {null_indicator_ids:,}")
print(f"Null entity IDs:        {null_entity_ids:,}")
print(f"Duplicate keys:         {duplicate_key_count:,}")
print(f"Invalid years:          {invalid_year_count:,}")
print(f"Out-of-range years:     {out_of_range_year_count:,}")

if actual_count != expected_count:
    raise RuntimeError(
        "Record count validation failed."
    )

if null_indicator_ids > 0:
    raise RuntimeError(
        "Null indicator IDs detected."
    )

if null_entity_ids > 0:
    raise RuntimeError(
        "Null entity IDs detected."
    )

print("\nAll pre-write Bronze validations passed.")

In [0]:
# Write Bronze Delta table

(
    observations_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        OBSERVATIONS_TABLE
    )
)

print(
    f"Saved Bronze observations to:"
    f"\n{OBSERVATIONS_TABLE}"
)

In [0]:
# Read-back validation

saved_df = spark.table(
    OBSERVATIONS_TABLE
)

saved_count = saved_df.count()

print(
    f"Downloaded records: {len(all_observations):,}"
)

print(
    f"Saved records:      {saved_count:,}"
)

if saved_count != len(all_observations):

    raise RuntimeError(
        "Bronze write row-count validation failed."
    )

saved_indicator_count = (
    saved_df
    .select("indicator_id")
    .distinct()
    .count()
)

if saved_indicator_count != indicator_count:

    raise RuntimeError(
        "Indicator count changed after Bronze write."
    )

print("Bronze observation write validated.")

In [0]:
# Final ingestion summary

non_null_count = (
    saved_df
    .filter(
        F.col("value").isNotNull()
    )
    .count()
)

null_count = (
    saved_count - non_null_count
)

overall_coverage_pct = round(
    (
        non_null_count
        / saved_count
    ) * 100,
    2
)

print("=" * 70)
print("WORLD BANK INDICATOR OBSERVATION INGESTION")
print("=" * 70)

print(
    f"Indicators:        {indicator_count}"
)

print(
    f"Entities:          {entity_count}"
)

print(
    f"Requested years:   {START_YEAR}-{END_YEAR}"
)

print(
    f"Total records:     {saved_count:,}"
)

print(
    f"Non-null values:   {non_null_count:,}"
)

print(
    f"Null values:       {null_count:,}"
)

print(
    f"Overall coverage:  {overall_coverage_pct}%"
)

print(
    f"Target:            {OBSERVATIONS_TABLE}"
)

print(
    "Layer:             Bronze"
)

print(
    "Status:            SUCCESS"
)